# Check genes

Summarize the total number of genes listed in each `name_s` output for the Su, SEA_AD, and Yost studies.

Each non-empty line in a `name_s_*.txt` file is one component written by `csv.writer`; the total gene count for a weight is the sum of comma-separated genes across all non-empty lines.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
study_dirs = {
    "Su": Path("Su_2020/output"),
    "SEA_AD": Path("SEA_AD/output"),
    "Yost": Path("Yost_2019/output"),
}


def parse_name_s(path: Path) -> dict:
    stem = path.stem.replace("name_s_", "")
    cell_type, weight = stem.rsplit("_", 1)

    total_genes = 0
    unique_genes = set()
    num_components = 0

    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue

        genes = [gene.strip() for gene in line.split(",") if gene.strip()]
        if not genes:
            continue

        num_components += 1
        total_genes += len(genes)
        unique_genes.update(genes)

    return {
        "cell_type": cell_type,
        "weight": float(weight),
        "num_components": num_components,
        "total_genes": total_genes,
        "unique_genes": len(unique_genes),
        "file": str(path),
    }


rows = []
for study, output_dir in study_dirs.items():
    for path in sorted(output_dir.glob("name_s_*.txt")):
        row = parse_name_s(path)
        row["study"] = study
        rows.append(row)

summary = pd.DataFrame(rows)
summary = summary[["study", "cell_type", "weight", "num_components", "total_genes", "unique_genes", "file"]]
summary = summary.sort_values(["study", "cell_type", "weight"]).reset_index(drop=True)
summary

,study,cell_type,weight,num_components,total_genes,unique_genes,file
0,SEA_AD,Astro,0.00,4,200,200,SEA_AD/output/name_s_Astro_0.0.txt
1,SEA_AD,Astro,0.25,10,188,188,SEA_AD/output/name_s_Astro_0.25.txt
2,SEA_AD,Astro,0.50,40,1452,1452,SEA_AD/output/name_s_Astro_0.5.txt
3,SEA_AD,Astro,1.00,40,1701,1701,SEA_AD/output/name_s_Astro_1.0.txt
4,SEA_AD,Astro,2.00,40,1736,1736,SEA_AD/output/name_s_Astro_2.0.txt
5,SEA_AD,Astro,5.00,40,1760,1760,SEA_AD/output/name_s_Astro_5.0.txt
6,SEA_AD,Astro,10.00,40,1716,1716,SEA_AD/output/name_s_Astro_10.0.txt
7,SEA_AD,Micro-PVM,0.00,2,10,10,SEA_AD/output/name_s_Micro-PVM_0.0.txt
8,SEA_AD,Micro-PVM,0.25,12,141,141,SEA_AD/output/name_s_Micro-PVM_0.25.txt
9,SEA_AD,Micro-PVM,0.50,40,983,983,SEA_AD/output/name_s_Micro-PVM_0.5.txt


In [3]:
summary.pivot_table(index=["study", "cell_type"], columns="weight", values="total_genes")

weight            0.00   0.25    0.50    1.00    2.00    5.00    10.00
study  cell_type                                                      
SEA_AD Astro      200.0  188.0  1452.0  1701.0  1736.0  1760.0  1716.0
       Micro-PVM   10.0  141.0   983.0  1503.0  1564.0  1618.0  1655.0
Su     cd4_BL       5.0   55.0   116.0   713.0   807.0   932.0  1047.0
       cd8_BL       6.0   66.0   191.0   574.0   714.0   839.0   949.0
Yost   CD8T       165.0   78.0   591.0   800.0   982.0  1021.0  1032.0